# Image pattern mining with Eclat on Google Colab

Use `image_patterns_local.ipynb` to run the same image analysis in a local Julia kernel.

Handwritten digits become transactions, and Eclat finds the sets of visual words that appear together in many images. This notebook encodes the UCI Semeion digits with a seeded patch codebook, mines the encoding at three thresholds with the team's Eclat, and shows the images that contain a pattern next to images that lack it. Run all cells on a CPU runtime. Colab has no Julia kernel, so Python cells install Julia 1.13.2, check its SHA-256, and call it.

The cells run in this order: install Julia and fetch the repository at an exact commit, instantiate the project, run the package tests (they include the hand-worked feature example), let Julia verify the cached Semeion file or fetch a verified copy from Kaggle or UCI, encode and mine the images, build the pattern features, show the tables and images, and pack the evidence. The whole run takes about ten minutes.

Set `ITEMSET_REVISION` in the first code cell to a full commit SHA to replay a run. The data and the results stay in `/content`, outside Git. Each run uses its own results directory and evidence ZIP. The final cells read the last completed run on disk. If the runtime has no run on disk, run the notebook from the start. `applications/image_patterns/README.md` explains the encoding and the outputs.

In [ ]:
import hashlib
import os
import re
import subprocess
import tarfile
import urllib.request
from pathlib import Path

REPOSITORY = "https://github.com/NgTHung/Itemset"
ITEMSET_REVISION = ""  # A full commit SHA replays a run; empty takes the head of the default branch.
JULIA_URL = "https://julialang-s3.julialang.org/bin/linux/x64/1.13/julia-1.13.2-linux-x86_64.tar.gz"
JULIA_SHA256 = "2ecdfc9890ab0a565fdd92404c915ba2a04a933dbd5d292cc224d13819d17d07"

root, julia_home = Path("/content/Itemset"), Path("/content/julia-1.13.2")
revision = ITEMSET_REVISION or os.environ.get("ITEMSET_REVISION") or subprocess.check_output(
    ["git", "ls-remote", "--exit-code", REPOSITORY, "HEAD"], text=True, timeout=60).split()[0]
if not re.fullmatch(r"[0-9a-f]{40}", revision):
    raise ValueError(f"ITEMSET_REVISION must be a full lowercase 40-character commit SHA, got {revision!r}")
if not root.exists():
    subprocess.run(["git", "clone", "--", REPOSITORY, str(root)], check=True)
subprocess.run(["git", "-C", str(root), "checkout", "--detach", revision], check=True)

if not julia_home.exists():
    archive, digest = Path("/content/julia-1.13.2.tar.gz"), hashlib.sha256()
    with urllib.request.urlopen(JULIA_URL, timeout=60) as source, archive.open("wb") as target:
        while chunk := source.read(1024 * 1024):
            target.write(chunk)
            digest.update(chunk)
    if digest.hexdigest() != JULIA_SHA256:
        archive.unlink()
        raise ValueError(f"Julia archive SHA-256 is {digest.hexdigest()}, expected {JULIA_SHA256}")
    julia_home.mkdir()
    with tarfile.open(archive) as package:
        package.extractall(julia_home, filter="data")
os.environ["PATH"] = f"{next(julia_home.glob('julia-*/bin'))}{os.pathsep}{os.environ['PATH']}"
os.environ["JULIA_NUM_THREADS"] = "1"
os.chdir(root)


def julia(*args):
    """Run Julia in the repository and stop the notebook on a nonzero exit, which a shell escape would not."""
    process = subprocess.Popen(["julia", "--startup-file=no", "--project=.", *args], stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace")
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait():
        raise RuntimeError(f"julia {' '.join(args)} exited with {process.returncode}")


print("Revision:", revision)
print(subprocess.check_output(["julia", "--version"], text=True).strip())

In [ ]:
julia("-e", "using Pkg; Pkg.instantiate()")

In [ ]:
# The suite checks the encoder, the miner on both variants, and the pattern features on a hand-worked case.
julia("test/runtests.jl")

In [ ]:
import json
from shutil import rmtree
from tempfile import mkdtemp

# Julia rejects existing output paths. Allocate a directory for each run and pass an unused child path.
output_parent = root / "results/raw"
output_parent.mkdir(parents=True, exist_ok=True)
run_dir = Path(mkdtemp(prefix="semeion-", dir=output_parent))
results = run_dir / "patterns"
try:
    julia("scripts/image_features.jl", "data/raw/semeion/semeion.data", results.relative_to(root).as_posix())
    record = {
        "results": results.relative_to(root).as_posix(),
        "repository": REPOSITORY,
        "julia_archive_sha256": JULIA_SHA256,
    }
    marker = run_dir / "latest.json"
    marker.write_text(json.dumps(record), encoding="utf-8")
    marker.replace(output_parent / "semeion-latest.json")
except BaseException:
    rmtree(run_dir)
    raise

## Reading the outputs

A patch codebook of 24 visual words is fitted on all 1,593 images with seed 42, and each image keeps the set of words it contains. A transaction loses where each word sits and how often it occurs. The whole dataset fits this codebook, so the counts describe this dataset and do not measure how well a classifier would generalize.

`thresholds.tsv` gives the number of itemsets at each minsup. `selected_patterns.tsv` gives the most supported singleton, the most supported combination, and the longest pattern, with the count of supporting images per digit. `image_features.tsv` has one row per image and one column per frequent pattern at the lowest threshold, 1 where the image contains the whole pattern, so a column sums to the support of its pattern. In the SVGs, blue rectangles locate the patches that matched a word, and a nonmatching image names the words it lacks.

In [ ]:
import json
import tomllib
from pathlib import Path

import pandas as pd
from IPython.display import SVG, display

root = globals().get("root", Path("/content/Itemset"))
latest = root / "results/raw/semeion-latest.json"
if not latest.is_file():
    raise FileNotFoundError("Run the Semeion mining cell to produce image-pattern results first.")
run = json.loads(latest.read_text(encoding="utf-8"))
results = root / run["results"]

print("manifest.toml: statistics of the encoding")
print(tomllib.loads((results / "manifest.toml").read_text(encoding="utf-8"))["statistics"])
for name in ("thresholds.tsv", "selected_patterns.tsv"):
    print(name)
    display(pd.read_csv(results / name, sep="\t"))
features = pd.read_csv(results / "image_features.tsv", sep="\t")
print("image_features.tsv:", features.shape[0], "images,", features.shape[1] - 2, "pattern columns")
for name in ("digits.svg", "visual_words.svg", "supporting-0.2.svg", "supporting-0.05.svg"):
    print(name)
    display(SVG(filename=str(results / name)))

In [ ]:
import hashlib
import json
import tomllib
import zipfile
from pathlib import Path

root = globals().get("root", Path("/content/Itemset"))
latest = root / "results/raw/semeion-latest.json"
if not latest.is_file():
    raise FileNotFoundError("Run the Semeion mining cell to produce image-pattern results first.")
run = json.loads(latest.read_text(encoding="utf-8"))
results = root / run["results"]

manifest = tomllib.loads((results / "manifest.toml").read_text(encoding="utf-8"))
files = sorted(p for p in results.rglob("*") if p.is_file())
record = {
    "repository": run["repository"],
    "revision": manifest["git_revision"],
    "julia": f"julia version {manifest['julia_version']}",
    "julia_archive_sha256": run["julia_archive_sha256"],
    "semeion_download_url": manifest["source_url"],
    "semeion_sha256": manifest["source_sha256"],
    "sha256": {p.relative_to(root).as_posix(): hashlib.sha256(p.read_bytes()).hexdigest() for p in files},
}
evidence = root.parent / f"itemset-{results.parent.name}-evidence.zip"
with zipfile.ZipFile(evidence, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in files:
        archive.write(path, path.relative_to(root))
    archive.writestr("result.json", json.dumps(record, indent=2))
source = record["semeion_download_url"] or "verified cache"
print(f"Semeion input: {source}, SHA-256 {record['semeion_sha256']}")
print(f"Evidence: {evidence} ({len(files)} files). Download it from the Files sidebar.")